# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [65]:
import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Optional

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [66]:
DATA_DIR = Path('data')   # adjust if your folder layout differs

job_postings = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(job_postings)} job postings, {len(golden)} golden entries.')
print('Sample job posting:', job_postings[0])

Loaded 10 job postings, 10 golden entries.
Sample job posting: {'id': 'j01', 'job_posting': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes job posting text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [67]:
def prompt_zero_shot(job_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [{
        'role': 'user',
        'content': f"""
Extract the following fields from this job posting:
- company name
- job role
- minimum years of experience


Job Posting:
{job_text}
"""
    }]


def prompt_few_shot(job_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    examples = """
Example 1:
Job Posting: "We are hiring a Data Analyst at Northwind Ltd. Preferred 2 years of experience."
Output:
{"company_name": "Northwind Ltd", "job_role": "Data Analyst", "minimum_years_experience": 2}

Example 2:
Job Posting: "Acme Corp is looking for a Senior Software Engineer with at least 5 years of experience."
Output:
{"company_name": "Acme Corp", "job_role": "Senior Software Engineer", "minimum_years_experience": 5}

"""
    return [{
        'role': 'user',
        'content': f"""
Use the examples below to extract the requested fields from the target job posting.

{examples}

Task:
Extract the following fields from the job posting below:
- company name
- job role
- minimum years of experience


Job Posting:
{job_text}
"""
    }]


def prompt_structured(job_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    schema = """
{
  "company_name": string | null,
  "job_role": string | null,
  "minimum_years_experience": "integer | null"
}
"""
    return [
        {
            'role': 'system',
            'content': """
You are an expert recruiting analyst who can can indepth analysis of the postings. Your job is to read a job posting and extract three fields:
1) company name
2) job role
3) minimum years of experience

Return a valid JSON object that matches this schema exactly:
""" + schema + """

"""
        },
        {
            'role': 'user',
            'content': f"""
Job Posting:
{job_text}
"""
        },
    ]


def prompt_cot(job_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            'role': 'user',
            'content': f"""
Extract these three fields from the job posting:
- company name
- job role
- minimum years of experience

1. Analyze the job posting carefully.
2. Extract these three fields from the job posting:
- company name
- job role
- minimum years of experience
3. Output valid JSON only with exactly these keys:
{{
  "company_name": "...",
  "job_role": "...",
  "minimum_years_experience": <integer or null>
}}


Job Posting:
{job_text}
"""
        }
    ]


STRATEGIES = {
    'Zero_shot': prompt_zero_shot,
    'Few_shot': prompt_few_shot,
    'Structured': prompt_structured,
    'CoT': prompt_cot,
}

## Step 3 — Async batching

Run all 10 job postings × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, job_posting_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [68]:

def parse_response(text: str) -> Optional[dict]:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    if text is None:
        return None

    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r'\s*```\s*$', '', cleaned, flags=re.IGNORECASE)

    # Try to isolate the JSON object if there is extra prose.
    match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
    if match:
        cleaned = match.group(0)

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        return None


async def run_one(strategy_name: str, job_posting: dict) -> dict:
    """Run one strategy on one job posting. Return a dict with all the captured fields."""
    strategy_fn = STRATEGIES[strategy_name]
    messages = strategy_fn(job_posting['job_posting'])

    t0 = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    elapsed = time.perf_counter() - t0

    raw_text = response.choices[0].message.content or ''
    parsed = parse_response(raw_text)

    prompt_tokens = response.usage.prompt_tokens if response.usage else 0
    completion_tokens = response.usage.completion_tokens if response.usage else 0
    cost_in = prompt_tokens * RATES[MODEL]['in']
    cost_out = completion_tokens * RATES[MODEL]['out']
    cost_usd = cost_in + cost_out

    return {
        'strategy': strategy_name,
        'job_posting_id': job_posting['id'],
        'raw_response': raw_text,
        'parsed_extraction': parsed,
        'latency_s': round(elapsed, 6),
        'cost_usd': round(cost_usd, 12),
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, job_posting)
        for strategy_name in STRATEGIES
        for job_posting in job_postings
    ]
    return await asyncio.gather(*tasks)


In [69]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'Zero_shot',
 'job_posting_id': 'j01',
 'raw_response': '- **Company Name:** Acme Corp\n- **Job Role:** Senior Software Engineer\n- **Minimum Years of Experience:** 5 years',
 'parsed_extraction': None,
 'latency_s': 1.675177,
 'cost_usd': 2.73e-05,
 'prompt_tokens': 70,
 'completion_tokens': 28}

## Step 4 — Score against the golden set

Three scores per (strategy × job posting) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [70]:
def score_accuracy(extracted: Optional[dict], gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if not isinstance(extracted, dict):
        return 0

    def normalise(value, field_name=None):
        if value is None:
            return None
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            value = int(value)
        value = str(value).strip().lower()
        if field_name == 'minimum_years_experience':
            value = value.replace('+', '').replace('years', '').strip()
        return value

    score = 0
    for field in ['company_name', 'job_role', 'minimum_years_experience']:
        gold_value = gold.get(field)
        extracted_value = extracted.get(field)
        if normalise(gold_value, field) == normalise(extracted_value, field):
            score += 1
    return score


async def score_llm_judge(job_text: str, extracted: Optional[dict], gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.

    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    extracted_text = json.dumps(extracted, ensure_ascii=False) if isinstance(extracted, dict) else 'None'
    gold_text = json.dumps(gold, ensure_ascii=False)

    judge_messages = [
        {
            'role': 'system',
            'content': 'You are a strict evaluator for a structured extraction task. Return only an integer from 1 to 4.'
        },
        {
            'role': 'user',
            'content': f"""
Judge the extracted output against the gold answer.

Rubric:
- 4 = all three fields correct
- 3 = two of three correct, no fabricated data
- 2 = one of three correct, or fabricated a field
- 1 = none correct or unparsable

Job Posting:
{job_text}

Gold JSON:
{gold_text}

Extracted JSON:
{extracted_text}

Return only a single integer 1, 2, 3, or 4 and nothing else.
"""
        },
    ]

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=judge_messages,
        temperature=TEMPERATURE,
    )
    text = response.choices[0].message.content or ''
    match = re.search(r'\b([1-4])\b', text)
    return int(match.group(1)) if match else 1

In [71]:
# Apply scoring to all 40 results
scored = []
for result in results:
    job_posting = next(s for s in job_postings if s['id'] == result['job_posting_id'])
    gold = golden[result['job_posting_id']]

    result['accuracy'] = score_accuracy(result.get('parsed_extraction'), gold)
    result['parse_success'] = 1 if result.get('parsed_extraction') is not None else 0
    result['llm_judge_score'] = await score_llm_judge(job_posting['job_posting'], result.get('parsed_extraction'), gold)
    scored.append(result)

print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [76]:
df = pd.DataFrame(scored)

strategy_order = list(STRATEGIES.keys())

summary = (
    df.groupby('strategy', sort=False)
    .agg({
        'accuracy': 'mean',
        'parse_success': 'mean',
        'llm_judge_score': 'mean',
        'cost_usd': 'sum',
        'latency_s': 'median',
    })
    .reindex(strategy_order)
)

summary['parse_success'] = summary['parse_success'] * 100
summary['llm_judge_score'] = summary['llm_judge_score'] * (25 / 4)
summary = summary.round({
    'accuracy': 2,
    'parse_success': 2,
    'llm_judge_score': 2,
    'cost_usd': 8,
    'latency_s': 3,
})

summary.columns = [
    'Accuracy (mean of 3)',
    'Parse rate (%)',
    'Judge score (out of 25)',
    'Total cost ($)',
    'Latency p50 (s)',
]
summary.style.format({
    'Accuracy (mean of 3)': '{:.2f}',
    'Parse rate (%)': '{:.2f}%',
    'Judge score (out of 25)': '{:.2f}',
    'Total cost ($)': '{:.8f}',
    'Latency p50 (s)': '{:.3f}',
})

,Accuracy (mean of 3),Parse rate (%),Judge score (out of 25),Total cost ($),Latency p50 (s)
strategy,,,,,
Zero_shot,0.00,0.00%,6.25,0.00029280,1.754
Few_shot,2.90,100.00%,23.75,0.00047760,1.724
Structured,2.80,100.00%,24.38,0.00042000,1.663
CoT,2.90,100.00%,25.00,0.00042150,1.697


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```